# Project1: Annual Inflation Rates Across Countries, 1960 - 2025

### Setup

#### Preparing environment

In [169]:
import pandas as pd
import csv

#### Loading data

The data I'll be working with is the World Bank inflation data, available at https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG

There are two csv files that I will be working on:

1. "API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_60170.csv": this includes yearly inflation rates from 1960 - 2025 for countries and regions.

In [170]:
inflation = pd.read_csv("API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_60170.csv", skiprows=4)
inflation.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111,NaN
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN
3,Africa Western and Central,AFW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,1.657818,NaN
4,Angola,AGO,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020,NaN


2. "Metadata_Country_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_60170.csv": this includes country information (region, income group). While this may be used for further analysis, this project uses it mainly to filter out region data from the inflation rate dataset.

In [171]:
country = pd.read_csv("Metadata_Country_API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_60170.csv")
country.head()

,Country Code,Region,IncomeGroup,SpecialNotes,TableName,Unnamed: 5
0,ABW,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,AFE,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern,NaN
2,AFG,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
3,AFW,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central,NaN
4,AGO,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN


### Cleaning & merging

Hereon, I will use the country dataset as base. To prepare for merging, I clean the country dataset.

Keep only country code, region, and income group for simplicity.

In [172]:
country = country[["Country Code", "Region", "IncomeGroup"]]
country.head()

,Country Code,Region,IncomeGroup
0,ABW,Latin America & Caribbean,High income
1,AFE,NaN,NaN
2,AFG,Middle East & North Africa,Low income
3,AFW,NaN,NaN
4,AGO,Sub-Saharan Africa,Lower middle income


In [173]:
country.isna().sum()

Country Code     0
Region          48
IncomeGroup     48
dtype: int64

Because all the NA values are in columns representing regions (e.g. AFE for African Eastern and Southern), I remove all these rows.

In [174]:
country = country.dropna()
country.info()

<class 'pandas.DataFrame'>
Index: 217 entries, 0 to 264
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Country Code  217 non-null    str  
 1   Region        217 non-null    str  
 2   IncomeGroup   217 non-null    str  
dtypes: str(3)
memory usage: 6.8 KB


Merging the two datasets.

In [175]:
inflation_countries = pd.merge(country, inflation, on="Country Code", how="left")
inflation_countries.head()

,Country Code,Region,IncomeGroup,Country Name,Indicator Name,Indicator Code,1960,1961,1962,1963,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,ABW,Latin America & Caribbean,High income,Aruba,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Middle East & North Africa,Low income,Afghanistan,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN
2,AGO,Sub-Saharan Africa,Lower middle income,Angola,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,...,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020,NaN
3,ALB,Europe & Central Asia,Upper middle income,Albania,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,...,1.986661,2.028060,1.411091,1.620887,2.041472,6.725203,4.758346,2.215874,2.146186,NaN
4,AND,Europe & Central Asia,High income,Andorra,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Move Country Name to the front, and remove irrelevant columns.

In [176]:
inflation_countries.insert(0, "Country Name", inflation_countries.pop("Country Name"))
inflation_countries = inflation_countries.drop(
    columns=["Indicator Name", "Indicator Code", "Unnamed: 70"]
)
inflation_countries.head()

,Country Name,Country Code,Region,IncomeGroup,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,Latin America & Caribbean,High income,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.931196,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,Middle East & North Africa,Low income,NaN,NaN,NaN,NaN,NaN,NaN,...,4.383892,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN
2,Angola,AGO,Sub-Saharan Africa,Lower middle income,NaN,NaN,NaN,NaN,NaN,NaN,...,30.694415,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020
3,Albania,ALB,Europe & Central Asia,Upper middle income,NaN,NaN,NaN,NaN,NaN,NaN,...,1.275432,1.986661,2.028060,1.411091,1.620887,2.041472,6.725203,4.758346,2.215874,2.146186
4,Andorra,AND,Europe & Central Asia,High income,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Checking available data.

In [177]:
inflation_countries.isna().sum()

Country Name      0
Country Code      0
Region            0
IncomeGroup       0
1960            154
               ... 
2021             38
2022             38
2023             40
2024             41
2025             48
Length: 70, dtype: int64

Lots of missing data observed, especially for early years.

Then, transform the dataset into long data, in which a row represents the inflation in a country in a year, to prepare for analysis.

In [178]:
# Pick year columns
year_columns = [col for col in inflation_countries.columns if col.isdigit()]

# Transform
inflation_long = inflation_countries.melt(
    id_vars=["Country Name", "Country Code", "Region", "IncomeGroup"],
    value_vars=year_columns,
    var_name="Year",
    value_name="Inflation",
)

# Drop missing values
inflation_long = inflation_long.dropna(subset=["Inflation"])

inflation_long.head()

,Country Name,Country Code,Region,IncomeGroup,Year,Inflation
10,Australia,AUS,East Asia & Pacific,High income,1960,3.728822
11,Austria,AUT,Europe & Central Asia,High income,1960,1.945749
14,Belgium,BEL,Europe & Central Asia,High income,1960,0.299459
16,Burkina Faso,BFA,Sub-Saharan Africa,Low income,1960,7.788162
25,Bolivia,BOL,Latin America & Caribbean,Lower middle income,1960,11.533153


Overview of the cleaned dataset.

In [179]:
inflation_long.info()

<class 'pandas.DataFrame'>
Index: 9151 entries, 10 to 14320
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Name  9151 non-null   str    
 1   Country Code  9151 non-null   str    
 2   Region        9151 non-null   str    
 3   IncomeGroup   9151 non-null   str    
 4   Year          9151 non-null   str    
 5   Inflation     9151 non-null   float64
dtypes: float64(1), str(5)
memory usage: 500.4 KB


So now we have a 9151 row dataset to work with. (which satisfies the project requirement of 1000 - 1000000 observations)

Now I export the processed dataset for future use.

In [180]:
inflation_long.to_csv("inflation_processed.csv", index=False)

### Analysis: computation

#### Using pandas

Read the data.

In [181]:
inflation_processed = pd.read_csv("inflation_processed.csv")

The mean, median & mode are:

In [182]:
rates = inflation_processed["Inflation"]

print("Mean:", rates.mean())
print("Median:", rates.median())
print("Mode:", rates.mode().iloc[0])

Mean: 22.445713897294638
Median: 4.70844436028192
Mode: 0.0


Observing mode is 0.0, which carry very limited information, I review and check the series.

In [183]:
rates.head(20)

0      3.728822
1      1.945749
2      0.299459
3      7.788162
4     11.533153
5      1.358696
6      1.438795
7      5.814076
8      0.790206
9      0.827301
10     1.536612
11     1.255230
12    -3.562192
13     1.676599
14     0.332829
15     1.177756
16     3.418127
17     4.139940
18     1.003576
19     1.477848
Name: Inflation, dtype: float64

In [184]:
(inflation_processed["Inflation"] == 0).sum()

np.int64(2)

So even the mode 0.0 appears only 2 times, which are as follows:

In [185]:
zero_rows = inflation_processed[inflation_processed["Inflation"] == 0]
zero_rows[["Country Name", "Year"]]

,Country Name,Year
256,Thailand,1963
2507,Jordan,1986


It could be the long digits make it unlikely for any value to appear more than once. Therefore, I keep only 2 decimal points for more information.

In [186]:
inflation_processed["Inflation"] = inflation_processed["Inflation"].round(2)
inflation_processed.head()

,Country Name,Country Code,Region,IncomeGroup,Year,Inflation
0,Australia,AUS,East Asia & Pacific,High income,1960,3.73
1,Austria,AUT,Europe & Central Asia,High income,1960,1.95
2,Belgium,BEL,Europe & Central Asia,High income,1960,0.30
3,Burkina Faso,BFA,Sub-Saharan Africa,Low income,1960,7.79
4,Bolivia,BOL,Latin America & Caribbean,Lower middle income,1960,11.53


Re-export the data.

In [187]:
inflation_processed.to_csv("inflation_rounded.csv", index=False)

Redo the descriptive stats.

In [188]:
rates = inflation_processed["Inflation"]

print("Mean:", rates.mean())
print("Median:", rates.median())
print("Mode:", rates.mode().iloc[0])

Mean: 22.445748005682443
Median: 4.71
Mode: 2.1


#### Using Python standard library

##### Load data

In [189]:
import csv

rates = []
with open("inflation_rounded.csv", newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        rates.append(float(row["Inflation"]))

In [190]:
##### Mean

In [191]:
def calculate_mean(data):
    sum = 0
    count = 0
    for inf in data:
        sum += inf
        count += 1
    mean = sum / count
    return mean


count = len(rates)
mean = calculate_mean(rates)
print(f"There are {count} observations, and the mean is {mean:.2f}.")


There are 9151 observations, and the mean is 22.45.


##### Median

In [192]:
def calculate_median(data):
    data_sorted = sorted(data)
    count = len(data)
    middle = count // 2
    if count % 2 == 0:
        median = (data_sorted[middle - 1] + data_sorted[middle]) / 2
    if count % 2 == 1:
        median = data_sorted[middle]
    return median


median = calculate_median(rates)

print(f"The median is {median:.2f}.")


The median is 4.71.


##### Mode

In [193]:
def calculate_mode(data):
    d = {}
    for value in data:
        if value not in d:
            d[value] = 1
        else:
            d[value] += 1
    mode = max(d, key=d.get)
    return mode


mode = calculate_mode(rates)

print(f"The mode is {mode:.2f}.")

The mode is 2.10.


### Data visualization

Here, I want to visualize how the mean inflation rate for each country is distributed.

Step 1: Calculate country means

In [194]:
country_mean = inflation_processed.groupby("Country Name")["Inflation"].mean()

country_mean.round(2).head()

Country Name
Afghanistan              5.25
Albania                 14.00
Algeria                  8.40
Angola                 319.80
Antigua and Barbuda      2.24
Name: Inflation, dtype: float64

Step 2: Visualize distribution

In [195]:
first = int(country_mean.min() // 5) * 5
last = int(country_mean.max() // 5) * 5

print("Distribution of average annual inflation rates by country")
print("Each * represents one country.\n")

for lower in range(first, last + 5, 5):
    upper = lower + 5
    count = ((country_mean >= lower) & (country_mean < upper)).sum()

    if count > 0:
        label = f"{lower}% – <{upper}%"
        bar = "*" * count
        print(f"{label:>12} | {bar} ({count})")

Distribution of average annual inflation rates by country
Each * represents one country.

    0% – <5% | ************************************************************************** (74)
   5% – <10% | ****************************************************** (54)
  10% – <15% | ********************* (21)
  15% – <20% | ********* (9)
  20% – <25% | * (1)
  25% – <30% | **** (4)
  30% – <35% | *** (3)
  35% – <40% | **** (4)
  40% – <45% | *** (3)
  45% – <50% | * (1)
  50% – <55% | ** (2)
  60% – <65% | * (1)
  65% – <70% | * (1)
  70% – <75% | ** (2)
  75% – <80% | * (1)
  85% – <90% | * (1)
 95% – <100% | * (1)
100% – <105% | * (1)
115% – <120% | ** (2)
155% – <160% | * (1)
195% – <200% | * (1)
200% – <205% | * (1)
215% – <220% | * (1)
260% – <265% | * (1)
315% – <320% | * (1)
675% – <680% | * (1)


Because most observations cluster below 15%, I change the step widths here for more clarity.

In [196]:
rate_heading = "Average annual inflation rate (%)"
count_heading = "Number of countries"
width = len(rate_heading)
divider = f"{'-' * width} | {'-' * len(count_heading)}"

print("Distribution of average annual inflation rates by country")
print(f"{rate_heading} | {count_heading}")
print(divider)

lower = 0
highest = country_mean.max()

while lower <= highest:
    if lower in (15, 50, 100):
        print(divider)

    if lower < 15:
        step = 1
    elif lower < 50:
        step = 5
    elif lower < 100:
        step = 10
    else:
        step = 100

    upper = lower + step
    count = ((country_mean >= lower) & (country_mean < upper)).sum()

    label = f"{lower}%–<{upper}%"
    bar = "*" * count
    print(f"{label.rjust(width)} | {bar} ({count})")

    lower = upper

print("\nEach * represents one country.")

Distribution of average annual inflation rates by country
Average annual inflation rate (%) | Number of countries
--------------------------------- | -------------------
                           0%–<1% | *** (3)
                           1%–<2% | *** (3)
                           2%–<3% | ********************* (21)
                           3%–<4% | ************************** (26)
                           4%–<5% | ********************* (21)
                           5%–<6% | ****************** (18)
                           6%–<7% | ********** (10)
                           7%–<8% | ************ (12)
                           8%–<9% | ******* (7)
                          9%–<10% | ******* (7)
                         10%–<11% | ****** (6)
                         11%–<12% | ****** (6)
                         12%–<13% | * (1)
                         13%–<14% | **** (4)
                         14%–<15% | **** (4)
--------------------------------- | -------------------
    

From the above, we can see that a considerable proportion of countries experience an average annual inflation of 2 - 6 %.

However, there is a noticeable long tail, with some countries experiencing dramatic inflation with over 100% average annual inflation.